In [ ]:
# DO NOT MODIFY THIS CELL

from abc import ABC, abstractmethod


class AbstractSearchInterface(ABC):
    """
    Abstract class to support search/insert operations (plus underlying data structure)

    """

    @abstractmethod
    def insertElement(self, element):
        """
        Insert an element in a search tree
            Parameters:
                    element: string to be inserted in the search tree (string)

            Returns:
                    "True" after successful insertion, "False" if element is already present (bool)
        """

        pass

    @abstractmethod
    def searchElement(self, element):
        """
        Search for an element in a search tree
            Parameters:
                    element: string to be searched in the search tree (string)

            Returns:
                    "True" if element is found, "False" otherwise (bool)
        """

        pass

In [ ]:
"""
An AVL tree (Adelson-Velsky and Landis) tree is another kind of balanced search tree with one extra invariant:
- For any node, |h(node.left) - h(node.right)| <= 1, where h : Node -> int, and h(Node) is the height of the subtree rooted at node.

This invariant — while relatively expensive to maintain — ensures an almost-perfect balance of the tree.
Combinations of left and right rotations are used to fix the tree to adhere to the rule.
"""


class AVLNode:
    """Represents a node in an AVL tree."""

    def __init__(self, key: str):
        self.key: str = key
        self.height: int = 0
        self.left: AVLNode | None = None
        self.right: AVLNode | None = None


class AVLTree(AbstractSearchInterface):
    """An AVL tree with insert, search and delete operations."""

    def __init__(self, root=AVLNode | None):
        self.root = root

    def __height_of(self, node: AVLNode | None) -> int:
        """Returns the number of edges from a given node to its furthest leaf."""
        # -1 since height(leaf) = 1 + max(-1, -1) = 0
        if node is None:
            return -1
        else:
            return 1 + max(
                node.left.height if node.left is not None else -1,
                node.right.height if node.right is not None else -1,
            )

    def __get_balance(self, node: AVLNode) -> int:
        """Return the balance of a given node. Equal to left_height - right_height."""
        # Balance of root = left height - right height
        return self.__height_of(node.left) - self.__height_of(node.right)

    def __rotate_left(self, node: AVLNode) -> AVLNode:
        """Performs a left rotation about a given node.
        Returns the new root of node's subtree.
        """
        # Rotate
        right = node.right
        rightLeft = node.right.left
        node.right = rightLeft
        right.left = node

        # Update heights
        node.height = self.__height_of(node)
        right.height = self.__height_of(right)

        return right  # This is the new root

    def __rotate_right(self, node: AVLNode) -> AVLNode:
        """Performs a right rotation about a given node.
        Returns the new root of node's subtree.
        """
        # Rotate
        left = node.left
        leftRight = node.left.right
        node.left = leftRight
        left.right = node

        # Update heights
        node.height = self.__height_of(node)
        left.height = self.__height_of(left)

        return left  # This is the new root

    def __balance_node(self, node: AVLNode) -> AVLNode:
        """Fixes a node that breaks the invariant using rotation operations.
        Pushes the imbalance up the tree (until it reaches the root).
        Code assumes node.left and node.right are balanced.
        """
        # The height of child nodes may have changed, so the height of node may have changed also
        node.height = self.__height_of(node)
        balance = self.__get_balance(node)
        # Case 1: left subtree too heavy and imbalance from left's left subtree
        if balance > 1 and node.left is not None and self.__get_balance(node.left) >= 0:
            return self.__rotate_right(node)
        # Case 2: left subtree too heavy and imbalance from left's right subtree
        if (
            balance > 1
            and node.left is not None
            and self.__get_balance(node.left) <= -1
        ):
            node.left = self.__rotate_left(node.left)
            return self.__rotate_right(node)
        # Case 3: mirror of case 1
        if (
            balance < -1
            and node.right is not None
            and self.__get_balance(node.right) <= 0
        ):
            return self.__rotate_left(node)
        # Case 4: mirror of case 2
        if (
            balance < -1
            and node.right is not None
            and self.__get_balance(node.right) > 0
        ):
            node.right = self.__rotate_right(node.right)
            return self.__rotate_left(node)
        # Otherwise, the tree is balanced (from the node down) so just return the unchanged node
        else:
            return node

    # --- Search ---

    def __search(self, element: str, node: AVLNode | None) -> bool:
        """Recursively searches the subtree rooted at node for element.
        Returns True if found.
        """
        if node is None:
            return False
        elif node.key == element:
            return True
        elif node.key > element:
            return self.__search(element, node.left)
        else:
            return self.__search(element, node.right)

    def searchElement(self, element: str) -> bool:
        found = self.__search(element, self.root)
        return found

    # --- Insert ---

    def __insert(self, element: str, node: AVLNode | None) -> AVLNode:
        """Recursively inserts an element into an AVL subtree rooted at node.
        Duplicates are assumed to have been filtered by the caller.
        """
        if node is None:
            return AVLNode(element)
        elif node.key > element:
            node.left = self.__insert(element, node.left)
        elif node.key < element:
            node.right = self.__insert(element, node.right)

        return self.__balance_node(node)

    def insertElement(self, element: str) -> bool:
        """Inserts an element into the AVL tree.
        Returns False if the element was already present; True otherwise.
        """
        if self.__search(element, self.root):
            return False
        self.root = self.__insert(element, self.root)
        return True

    # --- Delete ---

    def __get_minimum_key(self, node: AVLNode) -> str:
        """Returns the smallest key in the subtree rooted at node."""
        while node.left is not None:
            node = node.left
        return node.key

    def __delete(self, element: str, node: AVLNode) -> AVLNode:
        """Recursively deletes element from an AVL subtree rooted at node."""
        if node is None:
            return node

        if element < node.key:
            node.left = self.__delete(element, node.left)
        elif element > node.key:
            node.right = self.__delete(element, node.right)
        else:
            # If the node has exactly one child, remove it by replacing it with that child.
            # If it has no children, just remove it by returning None.
            if node.left is None or node.right is None:
                return node.left if node.right is None else node.right
            else:
                # Node matches but is not a leaf — replace with in-order successor and
                # delete the successor from the right subtree.
                node.key = self.__get_minimum_key(node.right)
                node.right = self.__delete(node.key, node.right)

        return self.__balance_node(node)

    def deleteElement(self, element: str) -> bool:
        """Deletes an element from an AVL tree.
        Returns False if the element was not present; True otherwise.
        """
        if not self.__search(element, self.root):
            return False
        self.root = self.__delete(element, self.root)
        return True